# 03 Rule Classification

Apply deterministic first-pass rules to the latest inventory output and produce a review table. Still dry-run only.

In [12]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_5.yaml'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH =', POLICY_PATH)


PROJECT_ROOT = c:\00_dev\SCH-FILE-ORGANIZER
POLICY_PATH = c:\00_dev\SCH-FILE-ORGANIZER\policy\SCH_fileserver_policy_v2_5.yaml


In [13]:
from datetime import datetime
import pandas as pd

from src.policy_loader import PolicyLoader
from src.rules import classify_inventory, save_rule_outputs

policy = PolicyLoader.from_file(POLICY_PATH)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [14]:
inventory_files = list(OUTPUT_DIR.glob('inventory_*.parquet'))
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'

# Pick the newest file by filesystem timestamp, not filename order.
latest_inventory = max(inventory_files, key=lambda p: p.stat().st_mtime)
print('Using inventory:', latest_inventory.name)
inv = pd.read_parquet(latest_inventory)
print('Rows:', len(inv))
print('Columns:', list(inv.columns))


Using inventory: inventory_HTL0049-01_OITYLO-KOKKALA_MANI_20260309_133149.parquet
Rows: 4241
Columns: ['scan_root', 'absolute_path', 'relative_path', 'parent_relative', 'filename', 'stem', 'suffix', 'size_bytes', 'modified_at', 'created_at', 'depth_segments', 'path_length', 'filename_length', 'is_hidden', 'is_symlink', 'top_segment', 'hash', 'is_duplicate_hash', 'duplicate_group_size']


## Schema note
`classify_inventory()` now backfills missing inventory fields such as `filename`, `suffix`, `parent_relative`, `path_length`, and `filename_length` if you loaded an older inventory parquet. For best consistency, rerun `02_inventory.ipynb` after policy or scanner changes.


In [15]:
classified = classify_inventory(inv, POLICY_PATH)
classified[['relative_path', 'rule_status', 'rule_reason', 'rule_confidence', 'proposed_relative_target']].head(5)

,relative_path,rule_status,rule_reason,rule_confidence,proposed_relative_target
0,.DS_Store,archive_or_delete_candidate,junk_system_or_temp_file,high,
1,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\.DS_Store,archive_or_delete_candidate,junk_system_or_temp_file,high,
2,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,move_to_special_folder,duplicate_exact_hash,high,
3,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,move_to_special_folder,duplicate_exact_hash,high,
4,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,move_to_special_folder,duplicate_exact_hash,high,


In [16]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'rule_classification_{stamp}'
csv_path, parquet_path = save_rule_outputs(classified, output_base)
print('CSV:', csv_path)
print('Parquet:', parquet_path)


CSV: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\rule_classification_20260309_133308\rule_classification_20260309_133308.csv
Parquet: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\rule_classification_20260309_133308\rule_classification_20260309_133308.parquet


In [17]:
classified.groupby('rule_status').size().sort_values(ascending=False).to_frame('count')

,count
rule_status,
move_to_special_folder,2564
review,1469
archive_or_delete_candidate,208


In [18]:
classified[classified['rule_status'] == 'archive_or_delete_candidate'][['relative_path', 'filename', 'rule_reason', 'proposed_relative_target']].head(10)

,relative_path,filename,rule_reason,proposed_relative_target
0,.DS_Store,.DS_Store,junk_system_or_temp_file,
1,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\.DS_Store,.DS_Store,junk_system_or_temp_file,
21,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\Thumbs.db,Thumbs.db,junk_system_or_temp_file,
31,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΟΙΚΟΔΟΜΙΚΗ ΑΔΕΙΑ\.DS_...,.DS_Store,junk_system_or_temp_file,
38,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΟΙΚΟΔΟΜΙΚΗ ΑΔΕΙΑ\ΣΧΕΔ...,.DS_Store,junk_system_or_temp_file,
68,ΓΗ-ΣΥΜΒΟΛΑΙΟ ΑΓΟΡΑΠΩΛΗΣΙΑΣ-ΜΙΣΘΩΣΗ ΓΗΣ\Thumbs.db,Thumbs.db,junk_system_or_temp_file,
80,ΕΙΣΦΟΡΕΣ ΚΑΙ ΟΙΚΟΝΟΜΙΚΑ ΑΠΟΔΕΙΚΤΙΚΑ\.DS_Store,.DS_Store,junk_system_or_temp_file,
94,ΠΡΟΜΗΘΕΥΤΕΣ\.DS_Store,.DS_Store,junk_system_or_temp_file,
95,ΠΡΟΜΗΘΕΥΤΕΣ\1. SONADO_ΠΡΟΜΗΘΕΥΤΕΣ\.DS_Store,.DS_Store,junk_system_or_temp_file,
96,ΠΡΟΜΗΘΕΥΤΕΣ\1. SONADO_ΠΡΟΜΗΘΕΥΤΕΣ\0. HABITATIO...,.DS_Store,junk_system_or_temp_file,


In [19]:
cols = [c for c in ['relative_path', 'filename', 'rule_reason', 'special_folder_target', 'proposed_relative_target'] if c in classified.columns]
classified[classified['rule_status'] == 'move_to_special_folder'][cols].head(10)

,relative_path,filename,rule_reason,special_folder_target,proposed_relative_target
2,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,01_SONADO_OITYLO_DHLWSH ANATHESHS.pdf,duplicate_exact_hash,_DUPLICATED,
3,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,02_SONADO_OITYLO_DHLWSH ANALHPSHS_STFNLAB OE.pdf,duplicate_exact_hash,_DUPLICATED,
4,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,03_SONADO_OITYLO_DHLWSH ANALHPSHS_GKA ENGINEER...,duplicate_exact_hash,_DUPLICATED,
5,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΓΚΡΙΣΗ ΕΦΟΡΕΙΑΣ ΑΡΧΑ...,01_SONADO_OITYLO_EGKRISH YPPO 2021.pdf,duplicate_exact_hash,_DUPLICATED,
6,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΛΕΓΧΟΣ ΔΟΜΗΣΗΣ\ΠΟΡΙΣ...,ΠΟΡΙΣΜΑ ΕΛΕΓΚΤΗ ΔΟΜΗΣΗΣ_01.03.2023_έχειν καλώς...,duplicate_exact_hash,_DUPLICATED,
7,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\01_SONADO IK...,01_SONADO IKE_TEXNIKH EKTHESH_SA.pdf,duplicate_exact_hash,_DUPLICATED,
8,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\02_SONADO IK...,02_SONADO IKE_TOPOGRAFIKO DIAGRAMMA_SA.pdf,duplicate_exact_hash,_DUPLICATED,
9,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\03_SONADO IK...,03_SONADO IKE_DIAGRAMMA DOMHSHS_SA.pdf,duplicate_exact_hash,_DUPLICATED,
10,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\04_SONADO IK...,04_SONADO IKE_KATOPSH A STATHMH_SA.pdf,duplicate_exact_hash,_DUPLICATED,
11,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\05_SONADO IK...,05_SONADO IKE_KATOPSH B STATHMH_SA.pdf,duplicate_exact_hash,_DUPLICATED,


In [20]:
classified[classified['rule_status'] == 'compliant_keep_review_path'][['relative_path', 'filename', 'parsed_phase', 'parsed_doc_type', 'default_folder_subpath', 'proposed_relative_target']].head(10)

,relative_path,filename,parsed_phase,parsed_doc_type,default_folder_subpath,proposed_relative_target


In [21]:
classified[classified['rule_status'] == 'review'][['relative_path', 'filename', 'rule_reason', 'path_risk', 'filename_risk']].head(10)

,relative_path,filename,rule_reason,path_risk,filename_risk
22,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\ΠΙΣΙΝΑ\ΚΑΤΟΨ...,ΚΑΤΟΨΗ ΠΙΣΙΝΑΣ .pdf,filename_not_in_canonical_pattern,,
23,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\ΠΙΣΙΝΑ\ΟΙΚ Α...,ΟΙΚ ΑΔΕΙΑ.pdf,filename_not_in_canonical_pattern,,
24,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\ΠΙΣΙΝΑ\ΟΨΕΙΣ...,ΟΨΕΙΣ&ΤΟΜΕΣ_ΠΙΣΙΝΑ.pdf,filename_not_in_canonical_pattern,,
25,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\ΠΙΣΙΝΑ\ΠΑΡΑΔ...,ΠΑΡΑΔΟΣΗ ΠΙΣΙΝΩΝ ΒΡΑΧΙΩΝΑ 4.pdf,filename_not_in_canonical_pattern,,
26,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\ΠΙΣΙΝΑ\ΠΙΣΙΝ...,ΠΙΣΙΝΑ (1).pdf,filename_not_in_canonical_pattern,,
30,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\ΠΙΣΙΝΑ\Πισιν...,Πισινα.msg,filename_not_in_canonical_pattern,,
47,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΟΙΚΟΔΟΜΙΚΗ ΑΔΕΙΑ\ΣΧΕΔ...,ΟΙΚΟΔΟΜΙΚΗ ΑΔΕΙΑ - Συντόμευση.lnk,filename_not_in_canonical_pattern,,
51,ΑΣΦΑΛΙΣΗ ΕΡΓΟΥ\14_11_2022 - ΑΣΦΑΛΙΣΗ (ΑΣΤΙΚΗ Ε...,14_11_2022 - ΑΣΦΑΛΙΣΗ (ΑΣΤΙΚΗ ΕΥΘΥΝΗ) ΚΑΤΑΣΚΕΥ...,filename_not_in_canonical_pattern,,
52,ΑΣΦΑΛΙΣΗ ΕΡΓΟΥ\20221121 - SONADO - ΝΕΟ ΣΥΜΒΟΛΑ...,20221121 - SONADO - ΝΕΟ ΣΥΜΒΟΛΑΙΟ - ΤΕΧΝΙΚΩΝ Α...,filename_not_in_canonical_pattern,,
53,ΑΣΦΑΛΙΣΗ ΕΡΓΟΥ\20221121 - Σώμα Ασφαλιστηρίου S...,20221121 - Σώμα Ασφαλιστηρίου SONADO - INTERAM...,filename_not_in_canonical_pattern,,


In [11]:
classified.sort_values(['action_priority', 'relative_path']).head(10)

,scan_root,absolute_path,relative_path,parent_relative,filename,stem,suffix,size_bytes,modified_at,created_at,...,routing_basis,counterparty_rule_ok,counterparty_rule_reason,default_folder_subpath,current_folder_subpath,path_length_warning,filename_length_warning,path_risk,filename_risk,action_priority
121,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,21011_YDREYSH_PLANS.dwl,21011_YDREYSH_PLANS,.dwl,51,2023-03-01 14:05:28.639566660,2026-03-09 05:53:20.530429840,...,,False,not_applicable,,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,False,False,,,10
122,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,21011_YDREYSH_PLANS.dwl2,21011_YDREYSH_PLANS,.dwl2,217,2023-03-01 14:05:28.920873404,2026-03-09 05:53:20.537228107,...,,False,not_applicable,,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,False,False,,,10
126,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,Thumbs.db,Thumbs,.db,6144,2023-02-13 12:22:05.263608456,2026-03-09 05:53:20.565987349,...,,False,not_applicable,,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,False,False,,,10
353,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,01_SONADO_OITYLO_KATOPSH A STATHMIS.bak,01_SONADO_OITYLO_KATOPSH A STATHMIS,.bak,664636,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.279737949,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
355,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,02_SONADO_OITYLO_KATOPSH B STATHMIS.bak,02_SONADO_OITYLO_KATOPSH B STATHMIS,.bak,9554920,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.290498972,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
357,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,03_SONADO_OITYLO_KATOPSH C STATHMIS.bak,03_SONADO_OITYLO_KATOPSH C STATHMIS,.bak,26347123,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.307997704,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
359,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,04_SONADO_OITYLO_KATOPSH D STATHMIS.bak,04_SONADO_OITYLO_KATOPSH D STATHMIS,.bak,14532003,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.335934401,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
542,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΔΙΑΦΟΡΑ\ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,04_DESIGN_ENGINEERING/ΔΙΑΦΟΡΑ/ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,01_SONADO_OITYLO_KATOPSH A STATHMIS.bak,01_SONADO_OITYLO_KATOPSH A STATHMIS,.bak,664636,2022-11-02 15:31:59.000000000,2026-03-09 05:53:23.451033354,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΔΙΑΦΟΡΑ/ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,False,False,,,10
544,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΔΙΑΦΟΡΑ\ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,04_DESIGN_ENGINEERING/ΔΙΑΦΟΡΑ/ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,02_SONADO_OITYLO_KATOPSH B STATHMIS.bak,02_SONADO_